# N-Gram Language Models

- **Overview**: In the textbook, language modeling was defined as the task of predicting the next word in a sequence given the previous words. In this assignment, we will focus on the related problem of predicting the next character in a sequence given the previous characters.

- **Learning goals**:
    - Understand how to compute language model probabilities using maximum likelihood estimation.
    - Implement basic smoothing, back-off and interpolation.
    - Have fun using a language model to probabilistically generate texts.
    - Use a set of language models to perform text classification.

- **Data**: We will provide you with training and development data that has been manually labeled. We will also give you a test set without labels. You will build a classifier to predict the labels on our test set. You can upload your classifier’s predictions to Gradescope. We will score its predictions and maintain a leaderboard showing whose classifier has the best performance.

- **Delieverables:** This assignment has several deliverables:
    - Your implementations for the functions in the skeleton code (this notebook)
    - Answers to all questions labeled as `Answer #.#` in a file named `report.pdf`
    - Your model’s output for the test set (your model will be ranked on a leaderboard against the other students’ outputs)

- **Grading**: We will use the auto-grading system called `PennGrader`. To complete the homework assignment, you should implement anything marked with `#TODO` and run the cell with `#PennGrader` note. **There will be no hidden tests in this assignment.** In other words, you will know your score once you finish all the `#TODO` and run all the `#PennGrader` tests!


## Recommended Readings
- [Language Modeling with N-grams](https://web.stanford.edu/~jurafsky/slp3/3.pdf). Dan Jurafsky and James H. Martin. Speech and Language Processing (3rd edition draft) .
- [A Bit of Progress in Language Modeling](https://arxiv.org/abs/cs/0108005). Joshua Goodman. Computer Speech and Language .
- [The Unreasonable Effectiveness of Character-level Language Models](http://nbviewer.jupyter.org/gist/yoavg/d76121dfde2618422139). Yoav Goldberg. Response to Andrej Karpathy's blog post. 2015.
- [Language Independent Authorship Attribution using Character Level Language Models](http://www.aclweb.org/anthology/E/E03/E03-1053.pdf). Fuchun Pen, Dale Schuurmans, Vlado Keselj, Shaojun Wan. EACL 2003.

## Setup: Dataset / Packages
- **Run the following cells without changing anything!**
- [Loading dataset from huggingface](https://huggingface.co/docs/datasets/v1.8.0/loading_datasets.html#from-local-files)

In [6]:
import math, random
from collections import Counter
from dill.source import getsource

In [7]:
## DO NOT CHANGE ANYTHING, JUST RUN

def get_class_source(cls):
    import re
    class_name = cls.__name__
    from IPython import get_ipython
    ipython = get_ipython()
    inputs = ipython.user_ns['In']
    pattern = re.compile(r'^\s*class\s+{}\b'.format(class_name))
    for cell in reversed(inputs):
        if pattern.search(cell):
            return cell
    return None

# Section 0: Generating N-Grams
- **Problem 0:** Write a function `ngrams(n, text)` that produces a list of all n-grams of the specified size from the input text. Each n-gram should consist of a 2-element tuple `(context, char)`, where the context is itself an n-length string comprised of the n characters preceding the current character. The sentence should be padded with n ~ characters at the beginning (we’ve provided you with `start_pad(n)` for this purpose). If n=0 , all contexts should be empty strings. You may assume that n≥0.

```
>>> ngrams(1, 'abc')
[('~', 'a'), ('a', 'b'), ('b', 'c')]

>>> ngrams(2, 'abc')
[('~~', 'a'), ('~a', 'b'), ('ab', 'c')]
```

In [9]:
### DO NOT CHANGE ###
def start_pad(n):
    ''' Returns a padding string of length n to append to the front of text
        as a pre-processing step to building n-grams '''
    return '~' * n
### DO NOT CHANGE ###

In [ ]:
def ngrams(n, text):
    ''' Returns the ngrams of the text as tuples where the first element is
        the length-n context and the second is the character '''

    padded_text = start_pad(n) + text
    return [(padded_text[i:i+n], padded_text[i+n]) for i in range(len(text))]

ngrams(2, 'carlos')

[('~~', 'c'), ('~c', 'a'), ('ca', 'r'), ('ar', 'l'), ('rl', 'o'), ('lo', 's')]

# Section 1: Creating an N-Gram Model

1.   List item
2.   List item


In this section, you will build a simple n-gram language model that can be used to generate random text resembling a source document. Your use of external code should be limited to built-in Python modules, which excludes, for example, `NumPy` and `NLTK`.


In [12]:
import math

class NgramModel(object):
    ''' A basic n-gram model using add-k smoothing '''

    def __init__(self, n, k = 0):
      # About `k`: you don't need to worry about it for problem 1.1, it is something you will need in problem 2.1
      self.k = k

      # contexts is a dictionary with the context as keys and corresponding value is a dictionary of count and
      # chars which is a dictionary of characters as key and it's count as value.
      # Please do not change the format of storing the counts and characters, the autograder uses this structure
      # For example:
      # if the context is 'ab' with count=3 and characters following it are 'a' and 'b' with counts 2 and 1 respectively
      # then the context dictionary would look like this:
      # {
      #   'ab' : {'count': 3,'chars': {'a' : 2, 'b' : 1}},
      #   'aa' : {...},
      #   ...
      # }
      self.n = n
      self.contexts = {}
      self.vocab = set()

    def get_vocab(self):
      ''' Returns the set of characters in the vocab '''
      return self.vocab

    def get_context(self):
      ''' Returns the dictionary for context '''
      return self.contexts

    def update(self, text):
      ''' Updates the model n-grams based on text '''
      padded = '~' * self.n + text

      for i in range(len(text)):
        context = padded[i:i + self.n]
        char = padded[i + self.n]

        # Update vocabulary
        self.vocab.add(char)

        # Initialize context entry if new
        if context not in self.contexts:
            self.contexts[context] = {'count': 0, 'chars': {}}

        # Increment context count
        self.contexts[context]['count'] += 1

        # Increment character count
        self.contexts[context]['chars'][char] = self.contexts[context]['chars'].get(char, 0) + 1

    def prob(self, context, char):
      ''' Returns the probability of char appearing after context '''
      if self.k == 0:
        # Original unsmoothed probability calculation
        if context not in self.contexts:
            return 1/len(self.get_vocab()) if self.get_vocab() else 0

        context_data = self.contexts[context]
        total_count = context_data['count']
        char_count = context_data['chars'].get(char, 0)
        return char_count / total_count if total_count > 0 else 0

      else:
        # Add-k smoothed probability calculation
        vocab_size = len(self.get_vocab())
        if vocab_size == 0:
            return 0

        if context not in self.contexts:
            # Unseen context - uniform probability
            return 1/vocab_size

        context_data = self.contexts[context]
        total_count = context_data['count']
        char_count = context_data['chars'].get(char, 0)

        # Apply add-k smoothing
        numerator = char_count + self.k
        denominator = total_count + (self.k * vocab_size)

        return numerator / denominator

    def random_char(self, context):
      ''' Returns a random character based on the given context and the
          n-grams learned by this model '''

      # Get the probability distribution for characters following this context
      vocab = sorted(self.get_vocab())
      r = random.random()
      cumulative_prob = 0.0

      # Iterate through sorted vocabulary
      for char in vocab:
          # Get probability of this character given the context
          char_prob = self.prob(context, char)
          cumulative_prob += char_prob

          # Check if random number falls in this character's probability range
          if r < cumulative_prob:
              return char

      # Fallback - return last character if numerical issues occur
      return vocab[-1] if vocab else ''

    def random_text(self, length):
      ''' Returns text of the specified character length based on the
          n-grams learned by this model '''

      # Initialize with starting context (n padding characters)
      context = start_pad(self.n) if self.n > 0 else ''
      result = []

      for _ in range(length):
          # Get next random character given current context
          next_char = self.random_char(context)
          result.append(next_char)

          # Update context for next iteration
          if self.n > 0:
              # For n>0, maintain n-length context by dropping oldest character
              context = (context + next_char)[-self.n:]

      return ''.join(result)

    def perplexity(self, text):
        ''' Returns the perplexity of text based on the n-grams learned by this model '''
        padded_text = '~' * self.n + text
        log_prob_sum = 0.0
        T = len(text)

        for i in range(T):
            context = padded_text[i:i + self.n]
            char = padded_text[i + self.n]
            prob = self.prob(context, char)

            if prob == 0:
                return float('inf')

            log_prob_sum += math.log(prob)

        avg_log_prob = log_prob_sum/T

        return math.exp(-avg_log_prob)


## 1.1 Initialization [3 points]
In the `NgramModel` class, write an initialization method `__init__(self, n, k)` which stores the order n of the model and initializes any necessary internal variables. Then write a method `get_vocab(self)` that returns the vocab (this is the `set` of all characters used by this model).

- **Problem 1.1:** finish `__init__(self, n, k)` and `get_vocab(self)` [3 points]

## 1.2 Update and Calculate Probabilities [15 points]
Write a method `update(self, text)` which computes the n-grams for the input sentence and updates the internal counts. Also write a method `prob(self, context, char)` which accepts an n-length string representing a context and a character, and returns the probability of that character occuring, given the preceding context. If you encounter a novel `context`, the probability of any given `char` should be 1/V where V is the size of the vocab.

- **Problem 1.2:** finish `update(self, text)` and `prob(self, context, char)` [15 points]

In [ ]:
# PennGrader - DO NOT CHANGE
all_test_cases = [
    (1, ['abab', 'abcd'], [('b','c'), ('a', '~')]),
    (2, ['ababa', 'abcdddd', 'babcdabag'], [('ab', 'a'), ('ad', 'b')]),
    (3, ['ababa', 'abcdddd', 'babcdabag'], [('ddd', 'd'), ('~~c', 'f')]),
    (2, ['abc'], [('~a', 'b'), ('~~', 'a')])
]

all_test_results = []
for test_data in all_test_cases:
    nm = NgramModel(test_data[0], 0)
    for text in test_data[1]:
        nm.update(text)
    vocab = nm.get_vocab()
    probs = []
    for prob_check in test_data[2]:
        prob = nm.prob(prob_check[0], prob_check[1])
        probs.append(prob)
    all_test_results.append((vocab, probs))

Correct! You earned 15/15 points. You are a star!

Your submission has been successfully recorded in the gradebook.


## 1.3 Random Character [6 points]

Write a method `random_char(self, context)` which returns a random character according to the probability distribution determined by the given context. Specifically, let $V=\langle v_1,v_2, \cdots, v_n \rangle$ be the vocab, sorted according to Python's natural lexicographic ordering, and let $r$ be a random number between 0 and 1. Your method should return the character $v_i$ such that:
$$
\sum_{j=1}^{i-1} P(v_j\ |\ \text{context}) \le r < \sum_{j=1}^i P(v_j\ | \ \text{context}).
$$
You should use a single call to the `random.random()` function to generate $r$, for example:
```
>>> m = NgramModel(0, 0)
>>> m.update('abab')
>>> m.update('abcd')
>>> random.seed(1)
>>> [m.random_char('') for i in range(25)]
['a', 'c', 'c', 'a', 'b', 'b', 'b', 'c', 'a', 'a', 'c', 'b', 'c', 'a', 'b', 'b', 'a', 'd', 'd', 'a', 'a', 'b', 'd', 'b', 'a']
```

- **Problem 1.3:** finish `random_char(self, context)` [6 points]

In [ ]:
# PennGrader - DO NOT CHANGE
corpus = ['ababa', 'abcdddd', 'babcdabag', 'dbaa', 'dbab']
nm = NgramModel(1, 0)
for text in corpus:
    nm.update(text)

random_seed_lst = [2, 3, 4]
random_char_lst = []
for seed in random_seed_lst:
    random.seed(seed)
    random_char = [nm.random_char('') for i in range(25)]
    random_char_lst.append(random_char)

Correct! You earned 6/6 points. You are a star!

Your submission has been successfully recorded in the gradebook.


## 1.4 Random Text [6 points]
In the `NgramModel` class, write a method `random_text(self, length)` which returns a string of characters chosen at random using the `random_char(self, context)` method. Your starting context should always be $n$ ~ characters, and the context should be updated as characters are generated. If $n=0$, your context should always be the empty string. You should continue generating characters until you've produced the specified number of random characters, then return the full string.

```python
>>> m = NgramModel(1, 0)
>>> m.update('abab')
>>> m.update('abcd')
>>> random.seed(1)
>>> m.random_text(25)
abcdbabcdabababcdddabcdba
```

- **Problem 1.4:** finish `random_text(self, length)` [6 points]

In [ ]:
# PennGrader - DO NOT CHANGE
corpus = ['ababa', 'abcdddd', 'babcdabag', 'dbaa', 'dbab']
nm = NgramModel(1, 0)
for text in corpus:
    nm.update(text)

random_seed_lst = [42, 43, 44]
random_text_lst = []
for seed in random_seed_lst:
    random.seed(seed)
    random_text = nm.random_text(25)
    random_text_lst.append(random_text)

Correct! You earned 6/6 points. You are a star!

Your submission has been successfully recorded in the gradebook.


## 1.5 Writing Shakespeare [6 points]

Now you can train a language model. We have provided this corpus of Shakespeare at `shakespeare_input.txt` (if you cannot find it, re-run the `Setup 2: Dataset / Packages` section).

We’ve also given you the function `create_ngram_model(model_class, path, n, k)` that will create and return an n-gram model trained on the entire file path provided and `create_ngram_model_lines(model_class, path, n, k)` that will create and return an n-gram model trained line-by-line on the file path provided. You should use the first for the Shakespeare file and the second for the city name files.

Try generating some Shakespeare with different order n-gram models. You should try running the following commands:

```python
>>> m = create_ngram_model(NgramModel, 'shakespeare_input.txt', 2)
>>> m.random_text(250)

>>> m = create_ngram_model(NgramModel, 'shakespeare_input.txt', 3)
>>> m.random_text(250)

>>> m = create_ngram_model(NgramModel, 'shakespeare_input.txt', 4)
>>> m.random_text(250)

>>> m = create_ngram_model(NgramModel, 'shakespeare_input.txt', 7)
>>> m.random_text(250)
```

What do you think? Is it as good as [1000 monkeys working at 1000 typewriters](https://www.youtube.com/watch?v=no_elVGGgW8)?

After generating a bunch of short passages, do you notice anything? *They all start with F!* In fact, after we hit a certain order, the first word is always *First*?  Why is that? Is the model trying to be clever? Is it saying *First, I will generate the word "First"*?  No, probably not.  Explain what is going on in your writeup.

- **Answer 1.5:** Generate Shakespear texts with at least 3 numbers of n and discuss on the results [6 points]
  - Write your response to this in your pdf

In [17]:
## DO NOT CHANGE ##
def create_ngram_model(model_class, path, n=2, k=0):
    ''' Creates and returns a new n-gram model trained on the city names
        found in the path file '''
    model = model_class(n, k)
    with open(path, encoding='utf-8', errors='ignore') as f:
        model.update(f.read())
    return model

def create_ngram_model_lines(model_class, path, n=2, k=0):
    ''' Creates and returns a new n-gram model trained on the city names
        found in the path file '''
    model = model_class(n, k)
    with open(path, encoding='utf-8', errors='ignore') as f:
        for line in f:
            model.update(line.strip())
    return model

In [18]:
# Example
m = create_ngram_model(NgramModel, 'shakespeare_input.txt', 12)
m.random_text(250)

"First Citizen:\nTherefore doth Lysander\nDeny your love, I'll take out no work on't.\n\nCASSIO:\n'Faith, I must; she'll rail in\nhis rope-tricks. I'll tell you largely of fair Hermia, ere I go;\nMy ear should catch cold shortly: there, take my coxcomb:\nwhy,"

In [19]:
## Answer 1.5 in your PDF

# Section 2: Smoothing, Perplexity, and Interpolation [30 points]

In this part of the assignment, you'll adapt your code in order to implement several of the  techniques described in [Section 3.5 of the Jurafsky and Martin textbook](https://web.stanford.edu/~jurafsky/slp3/3.pdf).

## 2.1 Smoothing [6 points]

Laplace Smoothing is described in section 3.5.1 of the book. Laplace smoothing adds one to each count (hence its alternate name *add-one smoothing*). Since there are *V* characters in the vocabulary and each one was incremented, we also need to adjust the denominator to take into account the extra V observations.

$$P_{Laplace}(w_i) = \frac{count_i + 1}{N+|V|}$$

A variant of Laplace smoothing is called *Add-k smoothing* or *Add-epsilon smoothing*. This is described in section *Add-k 3.5.2*.

```python
>>> m = NgramModel(1, 1)
>>> m.update('abab')
>>> m.update('abcd')
>>> m.prob('a', 'a')
0.14285714285714285
>>> m.prob('a', 'b')
0.5714285714285714
>>> m.prob('c', 'd')
0.4
>>> m.prob('d', 'a')
0.25
```

- **Problem 2.1**: Update your `NgramModel` code from Part 1 to implement add-k smoothing. (see comment of `## TODO 2.1`) [6 points]




In [ ]:
# PennGrader - DO NOT CHANGE
smoothing_test_cases = [
    (1, 1, ['abab', 'abcd'], ('b', 'a')),
    (2, 1, ['ababa', 'abcdddd', 'babcdabag'], ('ba', 'b')),
    (3, 2, ['ababa', 'abcdddd', 'babcdabag', 'dbaa', 'dbab'], ('aba', 'b'))
]
smoothing_test_results = []
for i, j, corpus, test_words in smoothing_test_cases:
    nm = NgramModel(i, j)
    for text in corpus:
        nm.update(text)
    smoothing_test_results.append(nm.prob(test_words[0], test_words[1]))

Correct! You earned 6/6 points. You are a star!

Your submission has been successfully recorded in the gradebook.


## 2.2 Perplexity [12 points]

How do we know whether a language model is good? There are two basic approaches:
1. Task-based evaluation (also known as **extrinsic** evaluation), where we use the language model as part of some other task, like automatic speech recognition, or spelling correcktion, or an OCR system that tries to covert a professor's messy handwriting into text.
2. **Intrinsic** evaluation: Intrinsic evaluation tries to directly evalute the goodness of the language model by seeing how well the probability distributions that it estimates are able to explain some previously unseen test set.

Here's what the textbook says:

> For an intrinsic evaluation of a language model we need a test set. As with many of the statistical models in our field, the probabilities of an N-gram model come from the corpus it is trained on, the training set or training corpus. We can then measure the quality of an N-gram model by its performance on some unseen data called the test set or test corpus. We will also sometimes call test sets and other datasets that are not in our training sets held out corpora because we hold them out from the training data.

> So if we are given a corpus of text and want to compare two different N-gram models, we divide the data into training and test sets, train the parameters of both models on the training set, and then compare how well the two trained models fit the test set.

> But what does it mean to "fit the test set"? The answer is simple: whichever model assigns a higher probability to the test set is a better model.

We'll implement the most common method for intrinsic metric of language models: *perplexity*.  The perplexity of a language model on a test set is the inverse probability of the test set, normalized by the number of characters. For a test set $W = w_1 w_2 ... w_N$:

$$Perplexity(W) = P(w_1 w_2 ... w_N)^{-\frac{1}{N}}$$

$$ = \sqrt[N]{\frac{1}{P(w_1 w_2 ... w_N)}}$$

$$ = \sqrt[N]{\prod_{i=1}^{N}{\frac{1}{P(w_i \mid w_1 ... w_{i-1})}}}$$

Now implement the `perplexity(self, text)` function in `NgramModel`. A couple of things to keep in mind:
1. Numeric underflow is going to be a problem, so consider using logs.
2. Perplexity is undefined if the language model assigns any zero probabilities to the test set. In that case your code should return positive infinity - `float('inf')`.
3. On your unsmoothed models, you'll definitely get some zero probabilities for the test set. To test you code, you should try computing perplexity on the training set, and you should compute perplexity for your language models that use smoothing and interpolation.

```python
>>> m = NgramModel(1, 0)
>>> m.update('abab')
>>> m.update('abcd')
>>> m.perplexity('abcd')
1.189207115002721
>>> m.perplexity('abca')
inf
>>> m.perplexity('abcda')
1.515716566510398
```

- **Problem 2.2**: implement the `perplexity(self, text)` function in `NgramModel` [6 points]

In [ ]:
# PennGrader - DO NOT CHANGE
ppl_test_cases = [
    (1, 0, ['abab', 'abcd'], 'abca'),
    (2, 0, ['ababa', 'abcdddd', 'babcdabag'], 'ba'),
    (3, 1, ['ababa', 'abcdddd', 'babcdabag', 'dbaa', 'dbab'], 'abcdddddddddd')
]
ppl_test_results = []
for i, j, corpus, test_words in ppl_test_cases:
    nm = NgramModel(i, j)
    for text in corpus:
        nm.update(text)
    ppl_test_results.append(nm.perplexity(test_words))

Correct! You earned 6/6 points. You are a star!

Your submission has been successfully recorded in the gradebook.


- **Answer 2.2:** Compare and discuss the perplexity for text that is similar and different from Shakespeare's plays. We provide you two dev text files, a New York Times article (`nytimes_article.txt`) and several of Shakespeare's sonnets (`shakespeare_sonnets.txt`); if you cannot find them, re-run `Setup 2: Dataset / Package` at the beginning. Also, feel free to experiment with your own text. [6 points]

In [22]:
## Answer 2.2 in your PDF

# Different Text: n=2,3,4 k=1
model = NgramModel(4, 1)
model.update("shakespeare_input.txt")
print(model.perplexity("nytimes_article.txt"))

# Similar Text: n=2,3,4 k=1
model = NgramModel(4, 1)
model.update("shakespeare_input.txt")
print(model.perplexity("shakespeare_sonnets.txt"))


14.05092926398084
10.139222241575542


## 2.3 Interpolation [12 points]

The idea of interpolation is to calculate the higher order n-gram probabilities also combining the probabilities for lower-order n-gram models. Like smoothing, this helps us avoid the problem of zeros if we haven't observed the longer sequence in our training data. Here's the math:

$$P_{interpolation}(w_i|w_{i−2} w_{i−1}) = \lambda_1 P(w_i|w_{i−2} w_{i−1}) + \lambda_2 P(w_i|w_{i−1}) + \lambda_3 P(w_i)$$

where $\lambda_1 + \lambda_2 + \lambda_3 = 1$.

We've provided you with another class definition `NgramModelWithInterpolation` that extends `NgramModel` for you to implement interpolation. If you've written your code robustly, you should only need to override the `get_vocab(self)`, `update(self, text)`, and `prob(self, context, char)` methods, along with the initializer.

The value of $n$ passed into `__init__(self, n, k)` is the highest order n-gram to be considered by the model (e.g. $n=2$ will consider 3 different length n-grams). Add-k smoothing should take place only when calculating the individual order n-gram probabilities, not when calculating the overall interpolation probability.

By default set the lambdas to be equal weights, but you should also write a helper function that can be called to overwrite this default. Setting the lambdas in the helper function can either be done heuristically or by using a development set, but in the example code below, we've used the default.

```python
>>> m = NgramModelWithInterpolation(1, 0)
>>> m.update('abab')
>>> m.prob('a', 'a')
0.25
>>> m.prob('a', 'b')
0.75

>>> m = NgramModelWithInterpolation(2, 1)
>>> m.update('abab')
>>> m.update('abcd')
>>> m.prob('~a', 'b')
0.4682539682539682
>>> m.prob('ba', 'b')
0.4349206349206349
>>> m.prob('~c', 'd')
0.27222222222222225
>>> m.prob('bc', 'd')
0.3222222222222222
```
- **Problem 2.3**: implement the `NgramModelWithInterpolation` class [6 points]

In [23]:
class NgramModelWithInterpolation(NgramModel):
    ''' An n-gram model with interpolation '''

    def __init__(self, n, k):
        super().__init__(n, k)
        # Create models for each order from 0 to n
        self.models = [NgramModel(i, k) for i in range(n+1)]
        # Initialize equal weights for interpolation
        self.lambdas = [1.0/(n+1) for _ in range(n+1)]

    def get_vocab(self):
        # Combine vocabularies from all models
        vocab = set()
        for model in self.models:
            vocab.update(model.get_vocab())
        return vocab

    def update(self, text):
        # Update each model with the text
        for model in self.models:
            model.update(text)

    def prob(self, context, char):
        # Calculate weighted sum of probabilities from all models
        total_prob = 0.0
        for i, model in enumerate(self.models):
            # For each model, get the appropriate context length
            model_context = context[-model.n:] if model.n > 0 else ''
            # Get probability from this model and weight it
            total_prob += self.lambdas[i] * model.prob(model_context, char)
        return total_prob

    def set_lambdas(self, lambdas):
        # Optional helper to set custom lambda weights
        if abs(sum(lambdas) - 1.0) > 1e-6:
            raise ValueError("Lambdas must sum to 1")
        if len(lambdas) != len(self.models):
            raise ValueError("Number of lambdas must match number of models")
        self.lambdas = lambdas

In [ ]:
# PennGrader - DO NOT CHANGE
interpolation_test_cases = [
    [(0, 1), ['abab'], ('a', 'a')],
    [(2, 1), ['abab', 'abcd'], ('~c', 'd')],
    [(2, 1), ['abeadab', 'abcd', 'eadb'], ('~a','b')],
    [(2, 1), ['abab', 'abcd', 'aaaaad'], ('bc', 'd')],
]

interpolation_test_results = []
for (n, k), corpus, (context, char) in interpolation_test_cases:
    nm_interpolation = NgramModelWithInterpolation(n, k)
    for text in corpus:
        nm_interpolation.update(text)
    interpolation_test_results.append(nm_interpolation.prob(context, char))

Correct! You earned 6/6 points. You are a star!

Your submission has been successfully recorded in the gradebook.


- **Answer 2.3**: Experiment with a few different lambdas and values of k and discuss the effects of smoothing and interpolation in terms of perplexity. Alternatively, you can also try different number of n. You may still use text files from 2.2. [6 points]



In [25]:
## Answer 2.3 in your PDF

# For Different k Values
k_values = [0.1, 0.5, 1.0, 2.0, 3.0]
for k in k_values:
  model = NgramModel(2, k)
  model.update("shakespeare_input.txt")
  uninterpolated  = model.perplexity("shakespeare_sonnets.txt")  # add-k smoothing acts as unsmoothed for small k
  print(f"Uninterpolated: {uninterpolated}")

  interp_model = NgramModelWithInterpolation(2, k)
  interp_model.update("shakespeare_input.txt")
  interpolated = interp_model.perplexity("shakespeare_sonnets.txt")
  print(f"Interpolated: {interpolated}\n")

lambda_sets = [
    [0.25, 0.25, 0.25, 0.25],
    [0.4, 0.3, 0.2, 0.1],
    [0.1, 0.2, 0.3, 0.4]
]
results_lambda = []

for lambdas in lambda_sets:
    model = NgramModelWithInterpolation(n=3, k=1)
    model.update("shakespeare_input.txt")
    model.set_lambdas(lambdas)
    pp = model.perplexity("shakespeare_sonnets.txt")
    results_lambda.append((lambdas, pp))

# Display
print(f"{'Lambdas':<35} {'Perplexity':<10}")
for lambdas, pp in results_lambda:
    print(f"{str(lambdas):<35} {pp:<10.4f}")


Uninterpolated: 4.622743498518729
Interpolated: 5.94276460674064

Uninterpolated: 7.82577283902799
Interpolated: 8.941961107090748

Uninterpolated: 9.603590178237155
Interpolated: 10.439933239944665

Uninterpolated: 11.19131604637002
Interpolated: 11.716200645627707

Uninterpolated: 11.932818716312909
Interpolated: 12.304870438042995

Lambdas                             Perplexity
[0.25, 0.25, 0.25, 0.25]            10.2594   
[0.4, 0.3, 0.2, 0.1]                10.6377   
[0.1, 0.2, 0.3, 0.4]                9.9419    


# Section 3: Text Classification using N-Grams [20 points + 5 bonus]
**No PennGrader in this section, see `Deliverables` for grading details**
## Overview
Language models can be applied to text classification. If we want to classify a text $D$ into a category $c \in C={c_1, ..., c_N}$. We can pick the category $c$ that has the largest posterior probability given the text. That is,

$$ c^* = arg max_{c \in C} P(c|D) $$

Using Bayes rule, this can be rewritten as:

$$ c^* = arg max_{c \in C} P(D|c) P(c)$$

If we assume that all classes are equally likely, then we can just drop the $P(c)$ term:

$$ = arg max_{c \in C} P(D|c)$$

Here $P(D \mid c)$ is the likelihood of $D$ under category $c$, which can be computed by training language models for all texts associated with category $c$. This technique of text classification is drawn from [literature on authorship identification](http://www.aclweb.org/anthology/E/E03/E03-1053.pdf), where the approach is to learn a separate language model for each author, by training on a data set from that author. Then, to categorize a new text D, they use each language model to calculate the likelihood of D under that model, and pick the  category that assigns the highest probability to D.


## Try it!
We have provided you training and validation datsets consisting of the names of cities. The task is to predict the country a city is in. The following countries are including in the dataset.

```
af	Afghanistan
cn	China
de	Germany
fi	Finland
fr	France
in	India
ir	Iran
pk	Pakistan
za	South Africa
```

We'll set up a leaderboard for the text classification task. **Your job is to configure a set of language models that perform the best on the text classification task.** We will use the `city names` dataset, which you should have already downloaded. The test set has one unlabeled city name per line.

Your code should output a file `test_labels.txt` with one two-letter country code per line.

Feel free to extend the `NgramModel` or `NgramModelWithInterpolation` when creating your language model. Possible ideas to consider and experiment with when creating your model are utilizing a special end-of-text character, trying a new method for determining the vocab, and improving how your model handles novel characters.

## Deliverables
- **Answer 3.1:** Train your best model on the given training data (located in `train` folder), and report accuracy on the given validation data (located in `val` folder) for your final model. In order to receive full credit, your model must be able to outperform all the baselines. [5 points]

- **Answer 3.2:** Describe the parameters you used for the final submission, such as n , k and interpolation. Discuss how you made the decision and why your combination have better results, we encourage using a table to display all validation accuracies of your experiment process. Be sure to include a detailed error analysis: identify several examples of cities on which your final model is making errors, discuss potential reasons. [15 points]

- **Answer 3.3:** Make predictions on the given test data (`cities_test.txt`) and upload `test_labels.txt` to Gradescope, which contains one two-letter country code per line. Please note that testing accuracy should be valid and outperform the baseline in order to receive full credits (included in 3.1). Top 3 will be granted 5 bonus points.

In [26]:
COUNTRY_CODES = ['af', 'cn', 'de', 'fi', 'fr', 'in', 'ir', 'pk', 'za']

In [27]:
# ## Answer 3.1 and 3.2 in your PDF ###

# Answer 3.1: Training and Validation
import os
from pathlib import Path

train_dir = Path("train")
val_dir = Path("val")

# n = 5
# k = 1
# lambdas = [0.5, 0.1, 0.1, 0.1, 0.1, 0.1]

n = 4
k = 1
lambdas = [0.5, 0.1, 0.1, 0.2, 0.1]

country_models = {}

for file in os.listdir(train_dir):
  if file.endswith(".txt"):
    country_code = file.replace(".txt", "")
    model = create_ngram_model_lines(NgramModelWithInterpolation, train_dir/file, n=n, k=k)
    model.set_lambdas(lambdas)
    country_models[country_code] = model

correct = 0
total = 0

for file in os.listdir(val_dir):
  if file.endswith(".txt"):
    true_country = file.replace(".txt", "")
    with open(val_dir / file, encoding='utf-8') as f:
      for city in f:
        city = city.strip()
        if not city:
          continue
        perplexities = {
          country: model.perplexity(city)
          for country, model in country_models.items()
        }
        predicted = min(perplexities, key=perplexities.get)
        correct += int(predicted == true_country)
        total += 1

accuracy = correct / total
print(f"Validation Accuracy: {accuracy:.4f}")


# Answer 3.2: Parameter Selection and Error Analysis
# Final model parameters
n = 4
k = 1
lambdas = [0.5, 0.1, 0.1, 0.2, 0.1]

# Train one model per country
final_models = {}
for file in os.listdir(train_dir):
  if file.endswith(".txt"):
    code = file.replace(".txt", "")
    model = create_ngram_model_lines(NgramModelWithInterpolation, train_dir / file, n=n, k=k)
    model.set_lambdas(lambdas)
    final_models[code] = model

# Track misclassified cities
misclassified = []
for file in os.listdir(val_dir):
  if file.endswith(".txt"):
    true_country = file.replace(".txt", "")
    with open(val_dir / file, encoding='utf-8') as f:
      for line in f:
        city = line.strip()
        if not city:
          continue
        perplexities = {code: model.perplexity(city) for code, model in final_models.items()}
        predicted = min(perplexities, key=perplexities.get)
        if predicted != true_country:
          misclassified.append((city, true_country, predicted))

# Display results
import pandas as pd
df_misclassified = pd.DataFrame(misclassified, columns=[" City Name ", " True Label ", " Predicted Label "])
print(df_misclassified)


# # Answer 3.3: Test Predictions:

# Final model parameters
n = 4
k = 1
lambdas = [0.5, 0.1, 0.1, 0.2, 0.1]

# Train final models on the entire training set
final_models = {}
for file in os.listdir(train_dir):
    if file.endswith(".txt"):
        code = file.replace(".txt", "")
        model = create_ngram_model_lines(NgramModelWithInterpolation, train_dir / file, n=n, k=k)
        model.set_lambdas(lambdas)
        final_models[code] = model

# Load test city names with lenient decoding
test_cities_path = Path("cities_test.txt")
with open(test_cities_path, "r", encoding="utf-8", errors="ignore") as f:
    test_cities = [line.strip() for line in f if line.strip()]

# Predict labels
test_labels = []
for city in test_cities:
    perplexities = {code: model.perplexity(city) for code, model in final_models.items()}
    predicted = min(perplexities, key=perplexities.get)
    test_labels.append(predicted)

# Write output to file
with open("test_labels.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(test_labels))

print("\ntest_labels.txt has been generated with", len(test_labels), "predictions.")

Validation Accuracy: 0.6700
       City Name  True Label  Predicted Label 
0        bonabdan          ir               za
1            lord          ir               de
2         karrapu          ir               af
3       lysohirka          ir               fi
4      kar kandeh          ir               af
..            ...         ...              ...
292      shah mir          pk               ir
293   nazar garhi          pk               za
294    montgomery          pk               fr
295  bar bamakhel          pk               af
296        shamal          pk               af

[297 rows x 3 columns]

test_labels.txt has been generated with 900 predictions.


# Submission
### Congratulation on finishing your homework! Here are the deliverables you need to submit to GradeScope
- This notebook and py file: rename to `homework3.ipynb` and `homework3.py`. You can download the notebook and py file by going to the top-left corner of this webpage, `File -> Download -> Download .ipynb/.py`
- Your `report.pdf`
  - including answers to 1.5, 2.2, 2.3, 3.1, and 3.2
- `test_labels.txt` from `Section 3`